# Convolutional Neural Networks, Part 2: Training a CNN on CIFAR-10

CSCI 6379 · Topic 21.

Part 1 argued that convolution is the right structure for images and costs far fewer weights than a dense layer. This notebook is the evidence. We train several multilayer perceptrons and one small CNN on **CIFAR-10** under identical conditions (same optimizer, batch size, and number of epochs), and give the largest MLP **more parameters** than the CNN on purpose. The point is a head-to-head comparison: does structure beat size?

Run this on a GPU. In Colab: **Runtime -> Change runtime type -> GPU**.

## Setup

Standard imports. We pick the GPU if one is available and fix the seeds so runs are repeatable. CIFAR-10 downloads automatically the first time (about 170 MB).

In [ ]:
import os, time
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)
print("device:", dev)

## Load CIFAR-10

CIFAR-10 (Krizhevsky, Nair & Hinton, 2009) is 60,000 colour images at 32x32 pixels in ten mutually exclusive classes, split into **50,000 training** and **10,000 test** images, exactly 1,000 test images per class. Each image is 32 x 32 x 3 = 3,072 numbers.

We convert to tensors and normalize each channel by CIFAR-10's per-channel mean and standard deviation, then wrap the datasets in `DataLoader`s (shuffled mini-batches for training).

In [ ]:
tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

root = os.path.expanduser("~/data/cifar10")
tr = datasets.CIFAR10(root, train=True,  download=True, transform=tf)
te = datasets.CIFAR10(root, train=False, download=True, transform=tf)

train = DataLoader(tr, batch_size=128, shuffle=True,  num_workers=2)
test  = DataLoader(te, batch_size=512, shuffle=False, num_workers=2)

classes = tr.classes
print("classes:", classes)
print(f"train={len(tr)}  test={len(te)}")

## An MLP baseline

The MLP flattens each image to a 3,072-vector and stacks fully connected + ReLU layers. It has no notion that the input was ever a picture: shuffling all 3,072 pixel positions consistently across the whole dataset would not change its accuracy at all.

`mlp(*hidden)` builds an MLP with the given hidden widths, always ending in a 10-way output.

In [ ]:
def mlp(*hidden):
    layers, prev = [nn.Flatten()], 3 * 32 * 32
    for h in hidden:
        layers += [nn.Linear(prev, h), nn.ReLU()]
        prev = h
    layers += [nn.Linear(prev, 10)]
    return nn.Sequential(*layers)

# quick sanity check: parameter counts for the three MLP shapes
for name, m in [("mlp 1x512",  mlp(512)),
                ("mlp 2x1024", mlp(1024, 512)),
                ("mlp 3x2048", mlp(2048, 1024, 512))]:
    print(f"{name:11s} {sum(p.numel() for p in m.parameters()):>10,} params")

## A small CNN

Two convolutional blocks, then a dense head. Each conv uses `padding=1` so the convolution keeps the spatial size and only the `MaxPool2d(2)` layers halve it: 32x32 -> 16x16 -> 8x8. After two blocks we have 64 feature maps of 8x8, which flatten to 64*8*8 = 4,096 features feeding a 512-unit dense layer and a 10-way output.

The convolution has spatial structure **built in** for free, before training starts: every filter states that nearby pixels belong together and that a pattern means the same thing wherever it appears.

In [ ]:
class CNN(nn.Module):
    """Two conv blocks, padding=1 so conv keeps the size and only pooling halves it."""
    def __init__(self, p_drop=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),      # 32x32 -> 32x32
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                                 # -> 16x16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                                 # -> 8x8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p_drop),
            nn.Linear(64 * 8 * 8, 512), nn.ReLU(),
            nn.Dropout(p_drop),
            nn.Linear(512, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

print(f"cnn        {sum(p.numel() for p in CNN().parameters()):>10,} params")

## Training and evaluation loop

`evaluate` reports mean loss and accuracy over a loader. `train_model` runs Adam (learning rate 1e-3) with cross-entropy for a number of epochs and prints **test accuracy per epoch**. This is exactly the setup from the course experiment: same optimizer, same batch size, same number of epochs for every model, so only the architecture differs.

In [ ]:
def evaluate(m, loader):
    m.eval(); n = correct = 0; loss_sum = 0.0
    crit = nn.CrossEntropyLoss(reduction="sum")
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(dev), y.to(dev)
            o = m(x)
            loss_sum += crit(o, y).item()
            correct  += (o.argmax(1) == y).sum().item()
            n += y.numel()
    return loss_sum / n, correct / n

def train_model(build, epochs, log=True):
    torch.manual_seed(0)
    m = build().to(dev)
    nparam = sum(p.numel() for p in m.parameters())
    opt = optim.Adam(m.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    t0 = time.time()
    tra = tea = 0.0
    for ep in range(1, epochs + 1):
        m.train()
        for x, y in train:
            x, y = x.to(dev), y.to(dev)
            opt.zero_grad(); loss = crit(m(x), y); loss.backward(); opt.step()
        _, tra = evaluate(m, train)
        _, tea = evaluate(m, test)
        if log:
            print(f"  epoch {ep:2d}  train {tra:.3f}  test {tea:.3f}")
    return {"params": nparam, "train_acc": tra, "test_acc": tea,
            "minutes": round((time.time() - t0) / 60, 1)}

## The head-to-head comparison

Four models, all trained on CIFAR-10 with the same Adam optimizer (lr 1e-3), the same batch size, and the same number of epochs, differing only in architecture. The largest MLP is deliberately oversized: **8.9 million weights, more than four times the CNN's 2.2 million.** If depth or capacity were the deciding factor, it should win.

The course experiment uses **30 epochs**. That is heavy for a free Colab session (roughly 30-60 minutes across all four models), so `EPOCHS` is set lower below to keep the notebook interactive. Raise it toward 30 to reproduce the note's numbers more closely. Actual figures depend on the number of epochs and the seed.

In [ ]:
EPOCHS = 8   # course experiment uses 30; raise this to reproduce the note's figures

MODELS = {
    "mlp_1x512":  lambda: mlp(512),
    "mlp_2x1024": lambda: mlp(1024, 512),
    "mlp_3x2048": lambda: mlp(2048, 1024, 512),
    "cnn":        lambda: CNN(0.0),
}

summary = {}
for name, build in MODELS.items():
    print(f"[{name}]")
    summary[name] = train_model(build, EPOCHS)

print("\n" + "=" * 58)
print(f"{'model':12s} {'params':>12s} {'train':>8s} {'test':>8s}")
print("-" * 58)
for name, s in summary.items():
    print(f"{name:12s} {s['params']:>12,} {s['train_acc']*100:>7.1f}% {s['test_acc']*100:>7.1f}%")

In [ ]:
import matplotlib.pyplot as plt

names  = list(summary)
params = [summary[n]["params"] for n in names]
tests  = [summary[n]["test_acc"] * 100 for n in names]
colors = ["#888"] * (len(names) - 1) + ["#1f77b4"]

plt.figure(figsize=(7, 4))
plt.scatter(params, tests, c=colors, s=90)
for n, p, t in zip(names, params, tests):
    plt.annotate(n, (p, t), textcoords="offset points", xytext=(6, 4), fontsize=9)
plt.xscale("log")
plt.xlabel("parameters (log scale)")
plt.ylabel("test accuracy (%)")
plt.title("CIFAR-10: accuracy vs parameter count")
plt.grid(alpha=.3)
plt.show()

## What to expect

With the full **30-epoch** run, the course experiment reports these figures:

| model | parameters | train | **test** |
|---|---|---|---|
| MLP, 1 hidden | 1,578,506 | 87.7% | 52.2% |
| MLP, 2 hidden | 3,676,682 | 89.4% | 52.6% |
| MLP, 3 hidden | 8,921,610 | 91.7% | **54.0%** |
| **CNN** | **2,168,362** | 98.8% | **74.3%** |

Three readings:

- **Adding weights to an MLP barely helps.** Going from 1.6M to 8.9M parameters (a factor of 5.7) buys just **1.8 points**, 52.2% to 54.0%. The MLPs are not failing for lack of capacity.
- **The CNN wins by 20.2 points using 4.1x fewer weights than the largest MLP.** Structure beats size. The CNN also reaches **62.3% after a single epoch**, already better than any MLP achieves after thirty.
- **The CNN reaches 98.8% train against 74.3% test**, a 24.6-point gap. It has partly memorized. Closing that gap is the subject of **Part 3**.

Your own numbers will be lower than the table if you left `EPOCHS` small, and will shift a little with the seed, but the ordering (CNN far ahead, the three MLPs bunched together) shows up within just a few epochs.

## Closing: the wider CIFAR family

We stayed with plain CIFAR-10, but the moment you read the literature you meet its relatives, all built the same way:

- **CIFAR-100** — same 60,000 images at 32x32 from the same authors and source, but spread over **100 classes** with only **600 images each** (500 train, 100 test). The 100 fine classes are grouped into **20 superclasses**, so every image carries a fine label ("maple tree") and a coarse one ("trees"). With one tenth the examples per class and ten times as many classes, it is markedly harder: a model in the low 90s on CIFAR-10 might sit in the 70s on CIFAR-100. To try it, swap `datasets.CIFAR10` for `datasets.CIFAR100` and change the final layer to 100 outputs.
- **CIFAR-10.1** (Recht et al., 2018) — a fresh test set of about 2,000 new images drawn by the original collection procedure, built to ask whether classifiers that ace the standard test set actually generalize. Accuracy drops several points on it.
- **CINIC-10** — enlarges CIFAR-10 to 270,000 images by mixing in downsampled ImageNet pictures, a drop-in option when 60,000 is not enough.

Next: **Part 3** tackles the train/test gap this CNN exposed, using regularization such as dropout and data augmentation.